### Feature Extraction & Engineering

##### 1. Load the clean dataset
- Install & Import important libraries
- Using the cleaned dataset from /data

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer

In [3]:
df = pd.read_csv("./data/cleaned_synthetic_fraud_dataset.csv")
df.head()

,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
0,1,1119,2022-04-21T21:41:41,AU,mobile_app,gaming,547.24,78.0,3,3,0,VPN suspected
1,2,1166,2022-01-31T02:59:48,JP,web,luxury,1069.52,17.0,0,2,0,overnight shipping
2,3,1142,2024-09-23T04:12:18,CN,web,travel,305.73,16.0,4,1,0,No transaction note found
3,4,1060,2024-04-06T03:56:36,SG,pos_terminal,electronics,2096.71,76.0,2,2,0,high value item
4,5,1018,2022-01-20T14:39:36,US,pos_terminal,groceries,663.15,18.0,2,3,0,first time merchant


##### 2. Datetime-based / Temporal Features

- `txn_hour` → hour of the day
- `txn_dayofweek` → day of the week
- `txn_week` → ISO week number
- `is_weekend` → weekend indicator
- `is_night` → night-time transaction indicator

Extracts the hour, day, and week from the transaction timestamp.

These extraction features are useful for detecting unusual transaction times, whish common signal of fraud

In [4]:
df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'])

df['txn_hour'] = df['transaction_datetime'].dt.hour
df['txn_dayofweek'] = df['transaction_datetime'].dt.dayofweek
df['txn_week'] = df['transaction_datetime'].dt.isocalendar().week.astype(int)
df['is_weekend'] = (df['txn_dayofweek'] >= 5).astype(int)
df['is_night'] = df['txn_hour'].isin([0,1,2,3,4,5]).astype(int)

df[['transaction_datetime', 'txn_hour', 'txn_dayofweek', 'txn_week', 'is_weekend', 'is_night']].head()

,transaction_datetime,txn_hour,txn_dayofweek,txn_week,is_weekend,is_night
0,2022-04-21 21:41:41,21,3,16,0,0
1,2022-01-31 02:59:48,2,0,5,0,1
2,2024-09-23 04:12:18,4,0,39,0,1
3,2024-04-06 03:56:36,3,5,14,1,1
4,2022-01-20 14:39:36,14,3,3,0,0


##### 3. Transaction Gaps & Count:
- `txn_gap_minutes` → time since previous transaction (per customer)
- `txn_7d_count` → counts of transactions the customer did in the last 7 days.
- `txn_hour_count` → number of transactions by hour
- `txn_dayofweek_count` → number of transactions by day of week

These features help to captures behavioral pattern, like sudden bursts in transaction activity, which are often red flags.

In [5]:
df = df.sort_values(by=['customer_id','transaction_datetime'])

# Time since previous txn (minutes)
df['prev_txn_time'] = df.groupby('customer_id')['transaction_datetime'].shift(1)
df['txn_gap_minutes'] = (df['transaction_datetime'] - df['prev_txn_time']).dt.total_seconds() / 60
df['txn_gap_minutes'] = df['txn_gap_minutes'].fillna(-1)

# Rolling 7-day count per customer
df['txn_7d_count'] = (
    df.groupby('customer_id')
      .rolling('7D', on='transaction_datetime')
      .transaction_id.count()
      .reset_index(level=0, drop=True)
)

# Average transactions per hour/day for the customer
df['txn_hour_count'] = df.groupby(['customer_id','txn_hour'])['transaction_id'].transform('count')
df['txn_dayofweek_count'] = df.groupby(['customer_id','txn_dayofweek'])['transaction_id'].transform('count')

df[['customer_id', 'transaction_datetime', 'txn_gap_minutes', 'txn_7d_count', 'txn_hour_count', 'txn_dayofweek_count']].head(10)

,customer_id,transaction_datetime,txn_gap_minutes,txn_7d_count,txn_hour_count,txn_dayofweek_count
997,1000,2022-10-05 15:13:00,-1.000000,NaN,1,2
952,1000,2023-02-05 06:18:18,176585.300000,NaN,1,2
93,1000,2023-03-23 18:04:02,66945.733333,NaN,1,2
456,1000,2023-04-05 00:14:27,17650.416667,NaN,1,2
1198,1000,2023-07-13 02:02:46,142668.316667,NaN,1,2
837,1000,2023-10-08 07:42:45,125619.983333,NaN,1,2
1059,1001,2022-02-24 11:36:47,-1.000000,NaN,1,2
355,1001,2023-05-18 15:21:18,645344.516667,NaN,1,2
896,1001,2023-07-31 21:21:59,106920.683333,NaN,1,1
741,1001,2023-10-08 14:05:08,98923.150000,NaN,1,1


##### 4. Amount Features

- `customer_mean_amount` → average transaction amount per customer
- `customer_std_amount` → std dev transaction amount per customer
- `amount_z_customer` → z-score relative to customer's historical amounts

These are crucial for detecting transactions that are anomalously large or small compared to the customer's normal behavior.

In [6]:
df['customer_mean_amount'] = df.groupby('customer_id')['amount'].transform('mean')
df['customer_std_amount'] = df.groupby('customer_id')['amount'].transform('std') + 1e-6
df['amount_z_customer'] = (df['amount'] - df['customer_mean_amount']) / df['customer_std_amount']

df[['customer_id', 'amount', 'customer_mean_amount', 'customer_std_amount', 'amount_z_customer']].head()

,customer_id,amount,customer_mean_amount,customer_std_amount,amount_z_customer
997,1000,672.47,1931.283333,1662.07937,-0.757373
952,1000,2593.31,1931.283333,1662.07937,0.398312
93,1000,302.22,1931.283333,1662.07937,-0.980136
456,1000,4395.07,1931.283333,1662.07937,1.482352
1198,1000,579.99,1931.283333,1662.07937,-0.813014


##### 6.Risky Flag
- `is_risky_country` → flag for rare countries (heuristic)
- `low_device_trust` → flag for low device trust score

Adds domain informed heuristics that commonly correlate with fraud.

In [7]:
# Mark rare countries as risky (example heuristic)
risky_countries = df['country'].value_counts().tail(3).index
df['is_risky_country'] = df['country'].isin(risky_countries).astype(int)

# Low device trust indicator
df['low_device_trust'] = (df['device_trust_score'] < 0.4).astype(int)

df[['country', 'is_risky_country', 'device_trust_score', 'low_device_trust']].head()

,country,is_risky_country,device_trust_score,low_device_trust
997,US,0,23.0,0
952,US,0,84.0,0
93,US,0,70.0,0
456,US,0,24.0,0
1198,US,0,23.0,0


##### 7. Anomaly Flag
- `high_amount_z` → extremely unusual transaction amounts for that customer

In [8]:
# High anomaly indicators
df['high_amount_z'] = (df['amount_z_customer'] > 3).astype(int)  # extremely unusual amounts

df[['amount', 'amount_z_customer', 'high_amount_z']].head()

,amount,amount_z_customer,high_amount_z
997,672.47,-0.757373,0
952,2593.31,0.398312,0
93,302.22,-0.980136,0
456,4395.07,1.482352,0
1198,579.99,-0.813014,0


##### Final Feature Engineered Dataset
- Saved feature engineered dataset to `/data` for modeling

In [ ]:
df.head()

df.to_csv('./data/feature_engineered_dataset.csv', index=False)

,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,...,txn_gap_minutes,txn_7d_count,txn_hour_count,txn_dayofweek_count,customer_mean_amount,customer_std_amount,amount_z_customer,is_risky_country,low_device_trust,high_amount_z
997,998,1000,2022-10-05 15:13:00,US,mobile_app,luxury,672.47,23.0,2,3,...,-1.000000,NaN,1,2,1931.283333,1662.07937,-0.757373,0,0,0
952,953,1000,2023-02-05 06:18:18,US,web,gaming,2593.31,84.0,0,0,...,176585.300000,NaN,1,2,1931.283333,1662.07937,0.398312,0,0,0
93,94,1000,2023-03-23 18:04:02,US,pos_terminal,electronics,302.22,70.0,3,1,...,66945.733333,NaN,1,2,1931.283333,1662.07937,-0.980136,0,0,0
456,457,1000,2023-04-05 00:14:27,US,mobile_app,gaming,4395.07,24.0,0,2,...,17650.416667,NaN,1,2,1931.283333,1662.07937,1.482352,0,0,0
1198,1199,1000,2023-07-13 02:02:46,US,mobile_app,luxury,579.99,23.0,1,0,...,142668.316667,NaN,1,2,1931.283333,1662.07937,-0.813014,0,0,0
